# 8. The Mining Firm Case Study

In [2]:
import pulp as pl

def solve(model):
    model.solve(pl.PULP_CBC_CMD(msg=False))
    if pl.LpStatus[model.status] != "Optimal":
        raise RuntimeError(pl.LpStatus[model.status])

warehouses = ["Le Havre", "Saint-Nazaire", "Lyon"]
ports = warehouses[:2]
regions = ["R1", "R2", "R3", "R4"]
kinds = ["powder", "granules"]
demand = {"powder": [150000,50000,100000,70000], "granules": [50000,50000,50000,30000]}
capacity = dict(zip(warehouses, [300000,300000,200000]))
initial = dict(zip(warehouses, [50000,100000,0]))
inbound = dict(zip(warehouses, [2.5,3,2]))
outbound = dict(zip(warehouses, [[7,6,8,8],[8,9,10,12],[9,8,6,7]]))
production = {"powder":85, "granules":89}
valuation = {"powder":80, "granules":89}
model = pl.LpProblem("Mining_Firm", pl.LpMinimize)
a = {(w,k):pl.LpVariable(f"Alsace_{w}_{k}", lowBound=0) for w in warehouses for k in kinds}
n = {(w,k):pl.LpVariable(f"trains_{w}_{k}", lowBound=0, cat=pl.LpInteger) for w,k in a}
b = {w:pl.LpVariable(f"Gabon_{w}", lowBound=0) for w in ports}
v = {w:pl.LpVariable(f"boats_{w}", lowBound=0, cat=pl.LpInteger) for w in ports}
q = {w:pl.LpVariable(f"conversion_{w}", lowBound=0) for w in ports}
s = {(w,k):pl.LpVariable(f"stock_{w}_{k}", lowBound=0) for w in warehouses for k in kinds}
x = {(w,r,k):pl.LpVariable(f"delivery_{w}_{r}_{k}", lowBound=0) for w in warehouses for r in range(4) for k in kinds}
model += (pl.lpSum((production[k]+inbound[w]+3)*a[w,k] for w,k in a)
          + pl.lpSum(84*b[w]+80000*v[w]+5*q[w] for w in ports)
          + pl.lpSum(outbound[w][r]*x[w,r,k] for w,r,k in x)
          + pl.lpSum(.01*valuation[k]*s[w,k] for w,k in s))
model += pl.lpSum(a[w,"powder"] for w in warehouses) >= 100000
model += pl.lpSum(a[w,"powder"] for w in warehouses) <= 400000
model += pl.lpSum(a[w,"granules"] for w in warehouses) <= 200000
model += pl.lpSum(b.values()) <= 350000
for w,k in a:
    model += a[w,k] == 500*n[w,k]
for w in ports:
    model += b[w] <= 10000*v[w]
for w in warehouses:
    model += s[w,"powder"] == initial[w]+a[w,"powder"]+b.get(w,0)-q.get(w,0)-pl.lpSum(x[w,r,"powder"] for r in range(4))
    model += s[w,"granules"] == a[w,"granules"]+q.get(w,0)-pl.lpSum(x[w,r,"granules"] for r in range(4))
    model += pl.lpSum(x[w,r,k] for r in range(4) for k in kinds) <= capacity[w]
for r in range(4):
    for k in kinds:
        model += pl.lpSum(x[w,r,k] for w in warehouses) == demand[k][r]
solve(model)
print(f"Minimum monthly cost: EUR {pl.value(model.objective):,.2f}")
for w in warehouses:
    print(f"\n{w}")
    for k in kinds:
        print(f"  Alsace {k}: {a[w,k].value():,.0f} t; trains: {n[w,k].value():.0f}; ending stock: {s[w,k].value():,.0f} t")
    if w in ports:
        print(f"  Gabon powder: {b[w].value():,.0f} t; boats: {v[w].value():.0f}; conversion: {q[w].value():,.0f} t")
    for r in range(4):
        for k in kinds:
            if x[w,r,k].value() > 1e-6:
                print(f"  To {regions[r]}, {k}: {x[w,r,k].value():,.0f} t")
assert sum(var.value() for var in s.values()) < 1e-5

Minimum monthly cost: EUR 40,570,000.00

Le Havre
  Alsace powder: 100,000 t; trains: 200; ending stock: 0 t
  Alsace granules: 100,000 t; trains: 200; ending stock: 0 t
  Gabon powder: 0 t; boats: 0; conversion: 0 t
  To R1, powder: 50,000 t
  To R1, granules: 50,000 t
  To R2, powder: 50,000 t
  To R2, granules: 50,000 t
  To R4, powder: 50,000 t

Saint-Nazaire
  Alsace powder: 0 t; trains: 0; ending stock: -0 t
  Alsace granules: 0 t; trains: 0; ending stock: 0 t
  Gabon powder: 0 t; boats: 0; conversion: 0 t
  To R1, powder: 100,000 t

Lyon
  Alsace powder: 120,000 t; trains: 240; ending stock: 0 t
  Alsace granules: 80,000 t; trains: 160; ending stock: 0 t
  To R3, powder: 100,000 t
  To R3, granules: 50,000 t
  To R4, powder: 20,000 t
  To R4, granules: 30,000 t


Minimum monthly cost: **€40,570,000**.

Alsace produces **220,000 tonnes of powder** and **180,000 tonnes of granules**. No Gabon production or powder conversion is needed. All **150,000 tonnes of opening powder stock** are used. Detailed shipments are shown in the code output.